In [1]:
!pip install -q -U ultralytics roboflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
print(type(api_key))
rf = Roboflow(api_key=api_key)

# Fill these in from your Roboflow dataset URL / "Download Dataset" snippet
WORKSPACE, PROJECT, VERSION = ("project-ppltd", "shelf-qoj7f", 3)

dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolo26")  # yolov8 format works for YOLO26
DATA_DIR = dataset.location
print(DATA_DIR)

<class 'str'>
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Shelf-3 in yolo26:: 100%|██████████| 5573/5573 [00:03<00:00, 1611.92it/s]


/kaggle/working/Shelf-3


In [3]:
import yaml, os

with open(os.path.join(DATA_DIR, "data.yaml")) as f:
    cfg = yaml.safe_load(f)

cfg["path"] = DATA_DIR
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"

DATA_YAML = "/kaggle/working/data.yaml"
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)

print(cfg["names"], "| classes:", cfg.get("nc", len(cfg["names"])))

['shelves'] | classes: 1


In [4]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")   # n / s / m / l / x. Swap size as needed.

results = model.train(
    data=DATA_YAML,
    time=10,            # hours; overrides epochs and rescales the LR schedule to fit
    epochs=300,         # upper bound, ignored in favor of `time`
    patience=30,
    imgsz=640,
    batch=16,           # use -1 for auto batch size
    workers=4,
    device=[0, 1] if __import__("torch").cuda.device_count() > 1 else 0,  # T4 x2 -> both GPUs
    project="/kaggle/working/runs",
    name="shelf3_yolo26",
    exist_ok=True,
    cache=False,
    amp=True,
    plots=True,
    save_period=5,      # checkpoint every 5 epochs in case the session dies
)

# %% Cell 5: validate best weights on the test split
best = YOLO("/kaggle/working/runs/shelf3_yolo26/weights/best.pt")
metrics = best.val(data=DATA_YAML, split="test")
print(metrics.box.map50, metrics.box.map)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, eras

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/546      1.71G      1.143     0.9617    0.01074         86        640: 100% ━━━━━━━━━━━━ 123/123 2.5it/s 49.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.5it/s 2.4s
                   all        188       1744      0.902      0.886      0.916      0.644

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/594      1.71G      1.138     0.8542    0.01079         94        640: 100% ━━━━━━━━━━━━ 123/123 2.5it/s 48.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.0it/s 3.0s
                   all        188       1744      0.879      0.883      0.922      0.659

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/615      1.71G      1.137     0.7911    0.01061         75        640: 100% ━━━━━━━━━━━━ 